In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, "scripts")
from selection import cosine_distance_matrix, greedy_quantile
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import harmonypy, warnings
warnings.filterwarnings("ignore")

cells0 = pd.read_parquet("data/crc/cells.parquet").reset_index(drop=True)
mask = cells0.sample_type.isin(["tumor", "normal"]).to_numpy()
Z = np.load("data/crc/pca50.npy").astype(np.float32)[mask]
cells = cells0[mask].reset_index(drop=True).rename(columns={"sample_id": "pid"})
cells["y"] = (cells.sample_type == "tumor").astype(int)

sig = pd.read_parquet("data/crc/signatures_lf.parquet").rename(columns={"sample_id": "pid"})
sig = sig[sig.sample_type.isin(["tumor", "normal"])].reset_index(drop=True)
Scol = [c for c in sig.columns if c.startswith("s") and c[1:].isdigit()]
SIGARR = sig[Scol].to_numpy().astype(np.float32)
Dsig = cosine_distance_matrix(SIGARR)
sidx = {p: i for i, p in enumerate(sig.pid)}
inv = {i: p for p, i in sidx.items()}
don = cells.drop_duplicates("pid")[["pid", "study", "y"]].reset_index(drop=True)
study_of = dict(zip(don.pid, don.study))
pid_rows = {p: g.index.to_numpy() for p, g in cells.groupby("pid", observed=True)}
CPD = 120

def med_sigma(Zc, m=1500):
    Zc = np.asarray(Zc, np.float32)
    r = np.random.default_rng(0)
    s = Zc[r.choice(len(Zc), min(m, len(Zc)), replace=False)]
    return float(np.median(pdist(s)))

def rff(Zc, pid_arr, order, D=512):
    Zc = np.asarray(Zc, np.float32)
    r = np.random.default_rng(0)
    sigma = med_sigma(Zc)
    W = (r.standard_normal((Zc.shape[1], D)) / sigma).astype(np.float32)
    b = r.uniform(0, 2 * np.pi, D).astype(np.float32)
    phi = np.sqrt(2 / D) * np.cos(Zc @ W + b)
    prow = pd.Series(np.arange(len(order)), index=order).loc[pid_arr].to_numpy()
    S = np.zeros((len(order), D), np.float32)
    c = np.zeros(len(order))
    np.add.at(S, prow, phi)
    np.add.at(c, prow, 1.0)
    return S / c[:, None]

def cellrows_rows(pids):
    parts = []
    for p in pids:
        idx = pid_rows[p]
        parts.append(idx if len(idx) <= CPD else np.random.default_rng(0).choice(idx, CPD, replace=False))
    return np.concatenate(parts)

def evaluate(core, ref_pids, held_pid, do_harmony=True):
    pids = list(core) + list(ref_pids) + list(held_pid)
    rows = cellrows_rows(pids)
    Zs = Z[rows]
    meta = cells.iloc[rows][["pid", "study", "y"]]
    if do_harmony:
        ho = harmonypy.run_harmony(Zs, meta, ["study"], max_iter_harmony=10)
        Zc = np.asarray(ho.Z_corr, np.float32)
        Zs = Zc if Zc.shape[0] == len(meta) else Zc.T
    order = pd.unique(meta.pid.to_numpy())
    S = rff(Zs, meta.pid.to_numpy(), order)
    yb = meta.drop_duplicates("pid").set_index("pid").y
    ix = {p: i for i, p in enumerate(order)}
    tr = [ix[p] for p in list(core) + list(ref_pids)]
    te = [ix[p] for p in held_pid]
    sc = StandardScaler().fit(S[tr])
    clf = LogisticRegression(C=0.01, max_iter=5000).fit(sc.transform(S[tr]), yb.loc[list(core) + list(ref_pids)])
    return roc_auc_score(yb.loc[held_pid], clf.predict_proba(sc.transform(S[te]))[:, 1])

def setup_held(H, seed=0):
    r = np.random.default_rng(seed)
    cs = don[(don.y == 1) & (don.study != H)].study.value_counts()
    ns = don[(don.y == 0) & (don.study != H)].study.value_counts()
    TUMOR_SRC = cs.index[0]
    NORMAL_SRC = [s for s in ns.index if s != TUMOR_SRC][0]
    held_pid = don.pid[don.study == H].tolist()
    core = (list(r.choice(don.pid[(don.study == TUMOR_SRC) & (don.y == 1)], min(25, int(cs.iloc[0])), replace=False)) +
            list(r.choice(don.pid[(don.study == NORMAL_SRC) & (don.y == 0)], min(25, int(ns[NORMAL_SRC])), replace=False)))
    ref_tum = don.pid[(don.y == 1) & (~don.study.isin([H, TUMOR_SRC]))].tolist()
    ref_nor = don.pid[(don.y == 0) & (~don.study.isin([H, NORMAL_SRC]))].tolist()
    return core, held_pid, ref_tum, ref_nor

print("crc setup ok |", cells.pid.nunique(), "samples,", cells.study.nunique(), "studies")

In [ ]:
def order_random(reservoir, seed):
    return list(np.random.default_rng(seed).permutation(list(reservoir)))

def order_quantile(reservoir, q=0.8):
    pool = [sidx[p] for p in reservoir]
    cen = SIGARR[pool].mean(0)
    start = pool[int(np.argmin(np.linalg.norm(SIGARR[pool] - cen, axis=1)))]
    rest = [i for i in pool if i != start]
    return [inv[i] for i in [start] + greedy_quantile(Dsig, [start], rest, len(rest), q)]

def order_coverage(reservoir):
    pids = list(reservoir)
    st = {p: study_of[p] for p in pids}
    studies = list(set(st.values()))
    rows = {s: [sidx[p] for p in pids if st[p] == s] for s in studies}
    cent = {s: SIGARR[rows[s]].mean(0) for s in studies}
    gc = SIGARR[[sidx[p] for p in pids]].mean(0)
    sdist = {s: np.linalg.norm(cent[s] - gc) for s in studies}
    ntrim = 1 if len(studies) >= 4 else 0
    keep = sorted(studies, key=lambda s: sdist[s])[:len(studies) - ntrim]
    dord = {s: sorted([p for p in pids if st[p] == s], key=lambda p: np.linalg.norm(SIGARR[sidx[p]] - cent[s])) for s in keep}
    order, i = [], 0
    while any(len(dord[s]) > i for s in keep):
        for s in keep:
            if len(dord[s]) > i:
                order.append(dord[s][i])
        i += 1
    return order

In [ ]:
HELD_ALL = ["Chen_2024_Cancer_Cell", "Lee_2020_Nat_Genet", "Joanito_2022_Nat_Genet", "Uhlitz_2021_EMBO_Mol_Med", "Zhang_2020_Cell"]
KS = [2, 4, 6, 8, 12]
NSEED = {"coverage": 1, "quantile0.8": 1, "random": 12}
COLS = {"coverage": "#27ae60", "random": "#7f7f7f", "quantile0.8": "#8e44ad"}

def orders(name, reservoir, seed):
    if name == "random": return order_random(reservoir, seed)
    if name == "quantile0.8": return order_quantile(reservoir)
    return order_coverage(reservoir)

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=True); axes = axes.ravel()
wins = {"coverage": 0, "quantile0.8": 0, "n": 0}
for ax, H in zip(axes, HELD_ALL):
    core, held_pid, ref_tum, ref_nor = setup_held(H)
    res = {s: {k: [] for k in KS} for s in NSEED}
    for name in NSEED:
        for seed in range(NSEED[name]):
            oc, on = orders(name, ref_tum, seed), orders(name, ref_nor, seed)
            for k in KS:
                res[name][k].append(evaluate(core, list(oc[:k]) + list(on[:k]), held_pid))
    raw = evaluate(core, [], held_pid, do_harmony=False)
    harm0 = evaluate(core, [], held_pid, do_harmony=True)
    rmean = {k: np.mean(res["random"][k]) for k in KS}
    rci = {k: 1.96 * np.std(res["random"][k], ddof=1) / np.sqrt(NSEED["random"]) for k in KS}
    for name in ["coverage", "quantile0.8", "random"]:
        m = np.array([np.mean(res[name][k]) for k in KS])
        if name == "random":
            se = np.array([rci[k] for k in KS])
            ax.plot(KS, m, "-o", color=COLS[name], label=f"random (n={NSEED['random']})")
            ax.fill_between(KS, m - se, m + se, color=COLS[name], alpha=0.25)
        else:
            ax.plot(KS, m, "-o", color=COLS[name], label=name)
    for name in ["coverage", "quantile0.8"]:
        for k in KS:
            wins["n"] += 0.5
            if res[name][k][0] > rmean[k] + rci[k]: wins[name] += 1
    ax.axhline(raw, ls=":", color="black", label=f"raw {raw:.2f}")
    ax.axhline(harm0, ls="-.", color="0.6", label=f"harmony no-ref {harm0:.2f}")
    ax.set_title(f"held={H[:16]}"); ax.set_xlabel("refs/class"); ax.set_ylim(0.40, 1.0); ax.legend(fontsize=6)
    print(f"{H[:20]} @K12 | cov={res['coverage'][12][0]:.3f} quant={res['quantile0.8'][12][0]:.3f} rand={rmean[12]:.3f}+-{rci[12]:.3f}", flush=True)
axes[-1].set_visible(False)
plt.tight_layout(); plt.show()
print(f"above random upper-CI: coverage {wins['coverage']}/{int(wins['n'])}  quantile {wins['quantile0.8']}/{int(wins['n'])}")